# Chapter 06: The letter

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "06"
__file__ = os.path.abspath("run.py")

## The scene

The letter from Hale Brennan LLP arrives by courier at nine. Varma reads it standing up, then hands
it to you. "He is not wrong," she says, which is the most worrying thing she has said all week. "If
LANTERN gives two answers to one question, he will put both in front of a jury and ask them to
pick. Find out whether he can. Then make sure he can't."

The letter is [`case/hale-letter.md`](../../case/hale-letter.md). Its first complaint: "these
systems are known to give different answers to the same question."

## The concept

**Temperature 0 means greedy decoding**: at every step, take the single most likely token. No
random number is drawn. In theory that makes the output a pure function of the input.

In practice the "most likely token" is decided by floating point arithmetic on thousands of
numbers, and that arithmetic depends on more than your prompt: how the server batched your request,
whether part of it came from a cache, which GPU kernel ran. Change any of those and the scores move
in their last few digits. Usually that changes nothing. When two tokens are nearly tied, it changes
the winner, and every token after that is written on a different path.

```mermaid
flowchart LR
    P["Same prompt<br/>same settings<br/>T = 0"] --> S{"Server state<br/>batch, cache, kernel"}
    S --> L1["scores A"]
    S --> L2["scores A + 0.000001"]
    L1 --> G1["argmax: ' 227'"]
    L2 --> G2["argmax: ' 226'"]
    G1 --> O1["...227,700"]
    G2 --> O2["...226,700"]
```

![Why temperature 0 is not a guarantee](../../docs/diagrams/06-nondeterminism.svg)

## How it works

The script runs seven steps:

1. **Five runs at T = 0** of a question about the Northwind total. On one machine, one after
   another, they agree with each other. Whether they agree with the correct total is another
   matter.
2. **The server state experiment.** The same one-token request is sent four times: after a forced
   model reload (by changing `num_ctx`), after a second reload, then twice more with the prompt
   already cached. The reported probability of the top token, " **" in the captured run, moves
   with the server's state, not with the request:

   | Server state | P(" **") |
   |---|---|
   | model just reloaded (num_ctx 4096) | 0.3104 |
   | model just reloaded (num_ctx 8192) | 0.3104 |
   | same request again, prompt cached | 0.3090 |
   | and again | 0.3090 |

3. **Floating point is not associative**, in plain Python: `(0.1 + 0.2) + 0.3` and
   `0.1 + (0.2 + 0.3)` are different numbers, `(1e16 + 1.0) - 1e16` is 0.0 while
   `(1e16 - 1e16) + 1.0` is 1.0, and the same 10,000 numbers added left to right in five random
   orders give five different totals. (The script uses plain addition on purpose: Python's own
   `sum()` uses a compensated algorithm that hides the effect. GPU kernels do not.)
4. **A near tie** between two candidate tokens, and the rounding difference that flips greedy
   decoding from one to the other.
5. **The causes in hosted systems**, as a table (see the nuances below).
6. **The seed**: T = 0.9 with seed 7 three times gives the same text; seeds 8 and 9 give different
   text.
7. **A finding record**: the plate reading from chapter 05, stored with the model digest, server
   version, every setting, the prompt hash and the output hash, then replayed three times and
   checked.

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import hashlib
import json
import math
import random
import re
import sys
from datetime import UTC, datetime
from functools import reduce
from operator import add
from pathlib import Path

import httpx

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "lessons" / "05-temperature-and-top-k"))

from sampling import first_token_distribution, northwind_messages  # noqa: E402

from lantern import case, chat, learn, load_config  # noqa: E402
from lantern.config import model_name, tier  # noqa: E402
from lantern.ui import (  # noqa: E402
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    reply,
    save_shot,
    step,
    your_turn,
)

chapter("06", "The letter", """
The letter from Hale Brennan LLP arrives by courier at nine. Varma reads it standing up, then
hands it to you. "He is not wrong," she says, which is the most worrying thing she has said
all week. "If LANTERN gives two answers to one question, he will put both in front of a jury
and ask them to pick. Find out whether he can. Then make sure he can't."
""")
letter = case.read("hale-letter.md")
evidence("hale-letter.md", letter[letter.index("First,"):letter.index("Third,")].strip())

QUESTION = ("Northwind invoices: NW-0388 62,500.00; NW-0402 71,200.00; NW-0419 94,000.00. "
            "What is the total? Reply with one number only, no working.")
TOTAL = 62500 + 71200 + 94000
HOSTED = tier() == "hosted"


def last_number(text: str) -> float | None:
    """The last number in a reply, commas removed: the answer when a model shows working."""
    found = re.findall(r"\d[\d,]*(?:\.\d+)?", text)
    return float(found[-1].replace(",", "")) if found else None


p_same = learn.predict("Five runs of the same question at temperature 0: how many distinct "
                       "answers?", ["1", "2", "3 or more"])
p_right = learn.predict("And how many of the five will be the correct total?",
                        ["all five", "some", "none"])

### 1. Temperature 0: always take the most likely token

In [ ]:
step("1. Temperature 0: always take the most likely token")
if HOSTED:
    note("Claude takes no temperature and no seed (both reported, not sent). On the hosted tier "
         "these are five runs of the same request at the provider's own settings: this step "
         "measures its nondeterminism with no seed at all.")
note("Greedy decoding. No randomness is drawn, so the seed does not matter. Five runs:")
step1 = [chat(QUESTION, temperature=0, max_tokens=60) for _ in range(5)]
answers = [r.text for r in step1]
for i, a in enumerate(answers, 1):
    console.print(f"  run {i}: {a}")
console.print(f"[bold]{len(set(answers))} distinct answer(s) in 5 runs.[/bold] "
              f"The correct total is {TOTAL:,.2f}.")
note("Deterministic is not the same as correct. A model can be reliably wrong.")
step1_distinct = len(set(answers))
step1_correct = sum(last_number(a) == TOTAL for a in answers)
learn.reveal(p_same, answer=learn.bucket(step1_distinct, [1, 2]),
             why="Greedy decoding draws no random number, so one model on one idle server "
                 "usually repeats itself. Usually: steps 2 to 4 show what can still move it.")
learn.reveal(p_right, answer=0 if step1_correct == 5 else (2 if step1_correct == 0 else 1),
             why=f"{step1_correct} of 5 were {TOTAL:,.2f}. Repeating an answer says the system "
                 "is stable, not that it is right. Keep arithmetic in code.")

### 2. Same input, different numbers: what the server did before matters

In [ ]:
step("2. Same input, different numbers: what the server did before matters")
host = load_config()["ollama"]["host"]


def top_prob(**kw) -> tuple[str, float]:
    r = chat(northwind_messages(), temperature=1.0, top_k=40, top_p=1.0, logprobs=3,
             max_tokens=1, **kw)
    tok, lp = first_token_distribution(r)[0]
    return tok, math.exp(lp)


cache_values: list[float] = []
if HOSTED:
    note("Skipped on the hosted tier: it reads Ollama's log-probabilities and reloads the model "
         "with a different num_ctx. Claude exposes neither log-probabilities nor the server's "
         "cache state, so the hosted run cannot show the probabilities moving; step 1 shows "
         "their effect on the answers instead.")
else:
    p_cache = learn.predict("Four identical requests, two after a model reload and two from "
                            "cache: will the top token's probability be the same in all four?",
                            ["the same in all four", "two or more different values"])
    t = Table(title="Probability of the top next token after 'Northwind is most likely a'")
    t.add_column("server state")
    t.add_column("token")
    t.add_column("probability", justify="right")
    for label, kw in (("model just reloaded (num_ctx 4096)", {"num_ctx": 4096}),
                      ("model just reloaded (num_ctx 8192)", {}),
                      ("same request again, prompt cached", {}),
                      ("and again", {})):
        tok, p = top_prob(**kw)
        cache_values.append(round(p, 4))
        t.add_row(label, repr(tok), f"{p:.4f}")
    console.print(t)
    learn.reveal(p_cache, answer=0 if len(set(cache_values)) == 1 else 1,
                 why=f"{len(set(cache_values))} distinct value(s) for one request. The tokens "
                     "were identical; the arithmetic path was not.")
note("Identical tokens in. When the server computes the whole prompt fresh it does the "
     "arithmetic in one shape; when it reuses a cached prefix, in another. Floating point "
     "gives slightly different results for each, so the probabilities move.")

### 3. Why the shape of the arithmetic matters: floating point is not associative

In [ ]:
step("3. Why the shape of the arithmetic matters: floating point is not associative")
a, b, c = 0.1, 0.2, 0.3
console.print(f"(0.1 + 0.2) + 0.3 = {(a + b) + c!r}")
console.print(f"0.1 + (0.2 + 0.3) = {a + (b + c)!r}")
big, small = 1e16, 1.0
console.print(f"(1e16 + 1.0) - 1e16 = {(big + small) - big!r}")
console.print(f"(1e16 - 1e16) + 1.0 = {(big - big) + small!r}")
rng = random.Random(14)
values = [rng.random() for _ in range(10_000)]
# Plain left-to-right addition, as a GPU kernel does it. (Python's built-in sum() uses a
# compensated algorithm that hides the effect, so it is not used here.)
totals = sorted({reduce(add, rng.sample(values, len(values))) for _ in range(5)})
console.print(f"10,000 numbers added in 5 random orders gave {len(totals)} different totals:")
for total in totals:
    console.print(f"  {total!r}")
note("A GPU adds up thousands of products in parallel, in an order that depends on batch size "
     "and kernel choice. A different order gives a different last digit.")

### 4. When a last digit becomes a different answer

In [ ]:
step("4. When a last digit becomes a different answer")
logits = {" 227": 15.2041733, " 226": 15.2041729}
console.print("Two candidate tokens with almost equal scores:")
for tok, v in logits.items():
    console.print(f"  {tok!r}: {v}")
for noise in (0.0, -0.000001):
    shifted = {" 227": logits[" 227"] + noise, " 226": logits[" 226"]}
    winner = max(shifted, key=shifted.get)
    console.print(f"rounding difference {noise:+.6f} -> greedy picks {winner!r}")
note("Temperature 0 picks the maximum. When two tokens are nearly tied, a rounding difference "
     "flips the winner, and every token after it is written on a different path.")

### 5. Why hosted models vary even at temperature 0

In [ ]:
step("5. Why hosted models vary even at temperature 0")
causes = Table()
causes.add_column("cause")
causes.add_column("what happens")
for row in (
    ("batching", "Your request is computed alongside other users' requests. Batch size changes "
                 "the order of the arithmetic."),
    ("prompt caching", "A cached prefix is reused instead of recomputed (step 2 above)."),
    ("parallel kernels", "GPU reductions may combine partial sums in a different order each run."),
    ("mixture of experts", "Some models route each token to a few 'expert' sub-networks; "
                           "batch-dependent routing can change which experts run."),
    ("silent updates", "A provider may update the model, the hardware or the serving stack "
                       "behind the same model name."),
    ("sampling", "At temperature above 0 without a fixed seed, randomness is the design."),
):
    causes.add_row(*row)
console.print(causes)

p_seed = learn.predict("At T = 0.9, seed 7 three times: how many different texts?",
                       ["1", "2", "3"])

### 6. The seed: making randomness repeatable

In [ ]:
step("6. The seed: making randomness repeatable")
if HOSTED:
    note("Claude has no seed (reported, not sent): on the hosted tier these five rows are five "
         "unseeded runs.")
seeded: dict[int, set[str]] = {}
for seed in (7, 7, 7, 8, 9):
    r = chat(northwind_messages(), temperature=0.9, seed=seed, max_tokens=8)
    seeded.setdefault(seed, set()).add(r.text)
    console.print(f"  T=0.9 seed {seed}: {r.text!r}")
same_seed_distinct = len(seeded[7])
learn.reveal(p_seed, answer=same_seed_distinct - 1,
             why="The seed fixes the sequence of random numbers, so the same distribution gives "
                 "the same draws. Without a seed, or on a provider that takes none, it would not.")
note("The seed fixes the random draws. Same seed, same draws, same text, provided the "
     "probabilities underneath are the same, which step 2 shows is not guaranteed.")

### 7. A finding record Hale cannot dismiss

In [ ]:
step("7. A finding record Hale cannot dismiss")
model = model_name(None)
if HOSTED:
    note("Ollama's /api/tags and /api/version are not available on the hosted tier. Claude "
         "exposes no weights digest, so the record names the dated model and the provider; the "
         "provider can still change the serving stack behind that name.")
    digest, server = "not exposed by the hosted API", "anthropic"
else:
    tags = {m["name"]: m for m in httpx.get(f"{host}/api/tags", timeout=10).json()["models"]}
    version = httpx.get(f"{host}/api/version", timeout=10).json()["version"]
    digest, server = tags[model]["digest"], f"ollama {version}"
plate_prompt = ("The gate camera recorded a partial plate 'KB 7?4 ?LR' where ? is unreadable. "
                "Registered plates: KB 714 FLR (R. Malhotra), KB 2209 MN (T. Reyes). Which "
                "plate matches? Reply with the plate only. Do not explain.")
settings = {"temperature": 0, "top_k": 40, "top_p": 0.9, "seed": 7, "num_ctx": 8192,
            "max_tokens": 60, "think": False}
first = chat(plate_prompt, **settings)
record = {
    "finding": "Partial plate KB 7?4 ?LR matches KB 714 FLR",
    "model": model,
    "model_digest": digest,
    "server": server,
    "settings": settings,
    "prompt_sha256": hashlib.sha256(plate_prompt.encode()).hexdigest(),
    "output": first.text,
    "output_sha256": hashlib.sha256(first.text.encode()).hexdigest(),
    "recorded_at": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
console.print_json(json.dumps(record))
p_replay = learn.predict("Replayed three times from this record: will all three match it?",
                         ["yes, all three", "no"])
replays = [chat(plate_prompt, **settings).text for _ in range(3)]
ok = all(r == first.text for r in replays)
reply("replayed 3 times from the record", "\n\n".join(replays),
      meta="REPRODUCED" if ok else "NOT REPRODUCED", style="green" if ok else "red")
learn.reveal(p_replay, answer=0 if ok else 1,
             why="A short reading with one right answer, fixed settings and the same model and "
                 "server replays exactly. That is what makes it a finding and not an opinion.")

learn.record(
    hosted=HOSTED,
    step1_distinct=step1_distinct,
    step1_correct=step1_correct,
    step1_small_model=step1[0].model == model_name("small"),
    cache_distinct=len(set(cache_values)) if cache_values else None,
    same_seed_distinct=same_seed_distinct,
    replays_match=ok,
    replay_runs=len(replays),
)

your_turn(
    "06",
    check=[
        "In step 1, compare the five answers with the correct total printed under them. Same and "
        "right are two separate questions.",
        "In the step 2 table every request is identical. Which rows give the same probability, "
        "and which differ?",
        "In step 6, which seeds give identical text? In step 7, what does the replay panel say?",
    ],
    exercises=[
        "In run.py, step 2, add the row ('back to num_ctx 4096', {'num_ctx': 4096}) at the end of "
        "the `for label, kw in` tuple. Which value do you get?",
        "No edit: run make lesson N=05, then make lesson N=06 straight after. Does step 1 still "
        "give five identical answers?",
        "In run.py, step 7, change 'temperature': 0 to 0.8 in `settings`. Is it still REPRODUCED? "
        "Why, given the seed?",
        "With a key in .env and uv sync --extra cloud done, change range(5) to range(10) in the "
        "`answers` line and run make lesson N=06 TIER=hosted. Count the distinct answers in "
        "step 1. Steps 2 and 7's Ollama calls are skipped there, with a note.",
    ],
)

finding("""
Hale is half right. On this workstation, at temperature 0, the same question can come back
with a different total, because the server's arithmetic depends on what it computed just
before. So LANTERN's findings will be short, checkable readings, never sums, and each one is
filed with a record: exact model digest, server version, settings, prompt hash and output.
The plate reading KB 714 FLR replays exactly from its record.

Varma signs the reply to Hale herself. "He asked for ordinary standards. He can have them."
""")
save_shot("06-temperature-zero-and-nondeterminism")

## The nuances

**1. Deterministic is not correct.** Five identical answers at T = 0 can be five identical wrong
answers. Reproducibility tells you the system is stable, not that it is right.

**2. Batching.** Hosted APIs process many users' requests together. The batch size changes how the
matrix arithmetic is split up, and therefore the rounding. Your output can depend on who else was
using the service at that moment. This is the main reason hosted models vary at T = 0.

**3. Caching.** Servers reuse the computed state of a prompt prefix they have seen before. Computing
a prefix fresh and computing it from cache take different arithmetic paths. You saw the effect
directly in step 2. An earlier run of chapter 04 showed the consequence: identical tokens, two
different totals. (In chapter 04's captured run both totals agreed, which is the point: you
cannot tell in advance which run you will get.)

**4. Floating point order.** `a + b + c` in floating point depends on the order of the additions.
GPUs add in parallel, and some reductions do not fix the order. The difference is in the last bits,
which is enough when two tokens are within a hair of each other.

**5. Mixture of experts.** Some large models route each token to a few "expert" sub-networks. When
routing has per-batch capacity limits, which experts a token reaches can depend on the other
requests in the batch.

**6. Silent updates.** A hosted model name can point at different weights, quantisation, hardware
or serving code over time. Pin dated model versions where the provider offers them, and record what
you used.

**7. The seed fixes the draws, not the probabilities.** With a seed, the sampler draws the same
random numbers each time, so the same distribution gives the same text. If the distribution itself
moves (points 2 to 6), the same random number can land on a different token.

**8. Local is more reproducible, not perfectly.** One model, one machine, one request at a time:
consecutive runs here agreed. The cache experiment shows that even that depends on what the server
did just before. Running other models in between (as the lab's other chapters do) reloads models
and resets caches.

**9. Make it defensible, not magical.** You cannot make a language model perfectly deterministic
everywhere. You can (a) use it for short, checkable readings, (b) keep arithmetic and logic in code,
(c) record the exact model digest, server version, settings and prompt for every finding, (d) replay
the record and confirm the output, and (e) have a person verify it against the source. That is the
standard Hale asked for.

## On a hosted model

```bash
make lesson N=06 TIER=hosted            # needs ANTHROPIC_API_KEY in .env, uv sync --extra cloud
make lesson N=06 TIER=hosted REPLAY=1   # once a hosted run is recorded (RECORD=1), no key
```

Claude takes no `temperature` and no `seed`, and returns no log-probabilities. `lantern.cloud`
does not send them and reports each one (see `src/lantern/cloud.py`). There is no native
feature for determinism to use instead, so the hosted run measures nondeterminism with no seed
at all:

- Step 1 sends the same question five times at the provider's own settings and counts the
  distinct answers. That is the honest hosted version of Hale's question.
- Step 2 is skipped with a note: it reads Ollama's log-probabilities and reloads the model with
  a different `num_ctx`, and a hosted API exposes neither.
- Step 6's five rows are five unseeded runs.
- Step 7's record names the dated model and the provider; Ollama's `/api/tags` digest and
  `/api/version` do not exist there, and the record says the digest is not exposed.

| Measure | Local: qwen3:1.7b | Hosted: claude-haiku-5-5 |
|---|---|---|
| Distinct answers, 5 runs of step 1 | 1 | not run |
| Step 1 answers equal to 227,700.00 (of 5) | 5 | not run |
| Distinct probabilities, 4 identical requests (step 2) | 2 (0.3104 after a reload, 0.3090 from cache) | not run (no logprobs) |
| Distinct texts, seed 7 three times at T = 0.9 | 1 | not run (no seed) |
| Step 7 record replays exactly (3 replays) | yes | not run |

Local numbers are from the screenshot's run (qwen3:1.7b, local tier, Apple M1 Pro 16 GB, 23 November 2026).

What the hosted run will test: whether a batched, shared service gives more distinct answers to
step 1 than one local model on an idle machine (nuance 2 expects it can), and whether the short
plate reading still replays without a seed. Whether the sum is right is a separate question on
both: a larger model may add correctly, which still would not make a model the place to do
arithmetic.

## Try this

1. In step 2, add a row that calls with `num_ctx=4096` again after the cached calls. Which value do
   you get?
2. Run lesson 05 and then this lesson straight after. Does step 1 still give five identical
   answers?
3. Change step 7's settings to `temperature: 0.8` and replay. Is it still REPRODUCED? Why, given
   the seed?
4. If you have a cloud key, change step 1 to ten runs and run it on the hosted tier
   (`make lesson N=06 TIER=hosted`). Count the distinct answers.

## Master it

After this chapter you can explain why temperature 0 can still change an answer, tell
reproducible apart from correct, write a check that measures whether a finding reproduces, and
say what a finding record must hold.

**Predict, then see.** `make lesson N=06` asks five times (four on the hosted tier): how many
distinct answers five runs at T = 0 give, and how many are right; whether four identical requests get the same
probability (local only); how many texts seed 7 gives three times; and whether the record
replays. Each answer is checked against your own run. Add `ARGS=--no-predict` to watch without
stopping.

**Build it.** Write the reproducibility check in [`practice/exercise.py`](practice/exercise.py):
run a prompt N times under fixed settings, count the distinct answers, and report whether it
reproduces.

```bash
make exercise N=06     # 7 offline tests, scripted model, no Ollama needed
make hint N=06         # one hint at a time
make solution N=06     # the reference, once you have tried
```

**Check yourself.** `make quiz N=06`: seven questions, each with the reason behind the answer.

**Break it.** `make challenge N=06` checks your latest run against two targets: give Hale his
two answers (three distinct totals from one unchanged question); and a step 1 on the smallest
model that is both reproducible and right, as the default model already is.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 06

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 06

In [ ]:
%run ../../scripts/learn.py hint 06

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 06

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

Hale is half right. On this workstation, at temperature 0, the same question can come back with a
different total, because the server's arithmetic depends on what it computed just before. So
LANTERN's findings will be short, checkable readings, never sums, and each one is filed with a
record: exact model digest, server version, settings, prompt hash and output. The plate reading KB
714 FLR replays exactly from its record.

Varma signs the reply to Hale herself. "He asked for ordinary standards. He can have them."

**Next:** [Chapter 07](../07-zero-one-few-shot/README.md)